# Notebook 03 — Allocation des couts non tagges

In [1]:
import pandas as pd
from pathlib import Path
df = pd.read_csv(Path('../../ressources/datasets/cur_sample.csv'), parse_dates=['usage_date'])
df_tagged = df[df['tag_team'].notna() & (df['tag_team'] != '')].copy()
df_untagged = df[df['tag_team'].isna() | (df['tag_team'] == '')].copy()
print(f'Couts tagges   : ${df_tagged.unblended_cost.sum():,.2f}')
print(f'Couts non tagges: ${df_untagged.unblended_cost.sum():,.2f}')
print(f'Total          : ${df.unblended_cost.sum():,.2f}')

Couts tagges   : $27,798.41
Couts non tagges: $6,150.44
Total          : $33,948.85


## Exercice 3 — Implémenter allocate_proportional()

In [2]:
def allocate_proportional(df_tagged: pd.DataFrame, df_untagged: pd.DataFrame) -> pd.DataFrame:
    """
    Repartit les couts non tagges proportionnellement aux couts tagges par equipe.
    Retourne un DataFrame avec le cout total alloue par equipe.
    """
    # Calcul du total tague par equipe
    team_tagged = df_tagged.groupby('tag_team')['unblended_cost'].sum()
    
    # Part de chaque equipe (en %)
    team_share = team_tagged / team_tagged.sum()
    
    # Total des couts non tagges a redistribuer
    total_untagged = df_untagged['unblended_cost'].sum()
    
    # Allocation proportionnelle
    team_allocated = total_untagged * team_share
    
    # Consolidation : tague + allocated
    result = pd.DataFrame({
        'tag_team': team_tagged.index,
        'cout_tague': team_tagged.values,
        'cout_alloue': team_allocated.values,
    })
    result['cout_total'] = result['cout_tague'] + result['cout_alloue']
    return result.sort_values('cout_total', ascending=False).reset_index(drop=True)

result = allocate_proportional(df_tagged, df_untagged)
print(result.to_string(index=False))

# Verification : la somme doit etre egale au total original
total_original = df['unblended_cost'].sum()
total_result = result['cout_total'].sum()
print(f'\nVerification conservation des couts :')
print(f'  Total original : ${total_original:,.2f}')
print(f'  Total result   : ${total_result:,.2f}')
print(f'  OK : {abs(total_original - total_result) < 0.01}')

tag_team  cout_tague  cout_alloue  cout_total
    data   7562.1310  1673.131834 9235.262834
platform   7426.4572  1643.113820 9069.571020
frontend   6669.0483  1475.536064 8144.584364
payments   6140.7764  1358.655183 7499.431583

Verification conservation des couts :
  Total original : $33,948.85
  Total result   : $33,948.85
  OK : True


## Comparaison des 3 methodes d'allocation

In [3]:
total_untagged = df_untagged['unblended_cost'].sum()
teams = df_tagged['tag_team'].unique()
n_teams = len(teams)

# Even split
even = df_tagged.groupby('tag_team')['unblended_cost'].sum() + total_untagged / n_teams

# Proportional (deja calcule)
proportional = result.set_index('tag_team')['cout_total']

comparison = pd.DataFrame({
    'even_split': even,
    'proportional': proportional,
})
comparison['difference_%'] = ((comparison['proportional'] - comparison['even_split']) / comparison['even_split'] * 100).round(1)
print(comparison.to_string())

           even_split  proportional  difference_%
tag_team                                         
data      9099.740225   9235.262834           1.5
frontend  8206.657525   8144.584364          -0.8
payments  7678.385625   7499.431583          -2.3
platform  8964.066425   9069.571020           1.2
